# Notebook 08b — Phase B v2: Pre-Registered Thresholds via Pearson-Gap Objective

## Why this notebook exists

08a tried to pre-register the three GREEN-edge thresholds using per-sample F1 against misclassification as the objective. It failed: misclassification rates on the Mondrian holdout were too low (0.3% NSL, 1.2% CIC, 20% UNSW) for F1 to have signal. The grid collapsed to corners.

**Diagnosis:** the health flag is not a misclassification predictor. It's a calibration-quality indicator. We need an objective that has signal regardless of misclassification rate.

## New objective

**Maximize the GREEN-vs-RED gap in mean per-cell SCTS-correctness Pearson, subject to constraints.**

Rationale: the flag is operationally a way to say "SCTS is reliable in GREEN cells, unreliable in RED cells." A good flag should produce a large gap between mean SCTS-correctness Pearson on GREEN cells vs RED cells.

**Constraints (to prevent degenerate solutions):**
- At least 10 cells flagged GREEN
- At least 10 cells flagged RED
- At least 60 of 90 cells have valid (computable) Pearson on the tuning partition

If no combo satisfies, we report this as a negative result and fall back to v2 defaults.

## Where Pearson is computed

Pearson per cell is computed on the **threshold-calibration sub-slice** of the Mondrian holdout (the same sub-slice as in 08a — 2.5K NSL, 2.7K UNSW, 4K CIC). This keeps tuning disjoint from eval.

Per-cell sample counts on tcal:
- Need at least 20 samples in cell on tcal for a stable Pearson
- Cells below threshold contribute NaN, excluded from gap computation

## Pre-registered grid (LOCKED)

Same grid as 08a:
- `T_GREEN_LO ∈ {0.001, 0.01, 0.05, 0.1, 0.2, 0.3}`
- `CLIFF_GREEN_HI ∈ {0.01, 0.05, 0.10, 0.20, 0.30}`
- `N_GREEN_LO ∈ {30, 50, 100, 200, 500}`

150 combos. RED edges fixed.

## Outputs

- `results/tables/phase_b_v2_grid_search.csv` — grid with gap + constraint compliance
- `results/tables/phase_b_v2_selected_thresholds.json` — locked thresholds
- `results/tables/scts_v2_calib_health_phaseb_v2.csv` — health flag with Phase B v2 thresholds
- `results/tables/scts_v2_canonical_with_health_phaseb_v2.csv` — augmented per-sample SCTS
- `results/tables/phase_b_v2_three_way_comparison.csv`
- `results/tables/bootstrap_cis_phaseb_v2.csv`
- `docs/phase_b_v2_findings.md`


In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)
print(f'Ready: {os.getcwd()}')

Mounted at /content/drive
Ready: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
# Imports and constants
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
from itertools import product
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
ARCHITECTURES = ['rf', 'xgb', 'dnn']
VARIANTS = ['5class_cw', '5class_smote']
MODELS_PER_DATASET = [f'{a}_{v}' for v in VARIANTS for a in ARCHITECTURES]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Pre-registered grid (SAME as 08a — locked)
GRID_T_GREEN_LO = [0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
GRID_CLIFF_GREEN_HI = [0.01, 0.05, 0.10, 0.20, 0.30]
GRID_N_GREEN_LO = [30, 50, 100, 200, 500]

# Fixed (NOT tuned) — same as v2
T_GREEN_HI_FIXED = 0.95
T_RED_HI = 0.999
T_RED_LO = 0.001
CLIFF_RED_LO = 0.20
CLIFF_THRESH = 0.95
N_RED_HI = 30

# Constraints on Phase B v2 selection
MIN_GREEN_CELLS = 10
MIN_RED_CELLS = 10
MIN_VALID_PEARSON_CELLS = 60  # out of 90 max
MIN_SAMPLES_FOR_PEARSON = 20  # per-cell threshold for tcal-partition Pearson

# Other
ALPHA_PRIMARY = 0.05
MIN_CALIB_MONDRIAN = 30
EPS = 1e-6
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

print('Grid: 150 combos')
print(f'Constraints: GREEN cells >= {MIN_GREEN_CELLS}, RED cells >= {MIN_RED_CELLS}, '
      f'valid-Pearson cells >= {MIN_VALID_PEARSON_CELLS}')

Grid: 150 combos
Constraints: GREEN cells >= 10, RED cells >= 10, valid-Pearson cells >= 60


In [3]:
# Conformal helpers
def split_conformal_threshold(probs, y_true, alpha):
    n = len(y_true)
    scores = 1.0 - probs[np.arange(n), y_true]
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(scores, q_level))

def mondrian_conformal_thresholds(probs, y_true, alpha, n_classes=5, min_calib=30):
    y_pred = probs.argmax(axis=1)
    marginal = split_conformal_threshold(probs, y_true, alpha)
    thresholds, fallback, n_per_class = {}, [], {}
    for c in range(n_classes):
        mask = (y_pred == c)
        n_c = int(mask.sum())
        n_per_class[c] = n_c
        if n_c < min_calib:
            thresholds[c] = marginal
            fallback.append(c)
        else:
            scores_c = 1.0 - probs[mask, :][np.arange(n_c), y_true[mask]]
            q = min(np.ceil((n_c + 1) * (1 - alpha)) / n_c, 1.0)
            thresholds[c] = float(np.quantile(scores_c, q))
    return thresholds, fallback, n_per_class

def component_3_safe(probs, y_pred, thresholds, eps=1e-9):
    n = len(y_pred)
    sample_thresh = np.array([thresholds[int(p)] for p in y_pred], dtype=np.float64)
    s = 1.0 - probs[np.arange(n), y_pred]
    c3 = np.zeros(n, dtype=np.float32)
    nonzero = sample_thresh > eps
    c3[nonzero] = np.clip(1.0 - s[nonzero] / sample_thresh[nonzero], 0.0, 1.0)
    zero_mask = ~nonzero
    c3[zero_mask] = (s[zero_mask] <= eps).astype(np.float32)
    return c3

print('Helpers ready.')

Helpers ready.


In [4]:
# Load Phase A artifacts
mondrian_holdout_idx = {}
strict_holdout_probs = {}
strict_test_probs = {}

for ds in DATASETS:
    cal_dir = Path(REPO) / 'calibrators' / ds
    mondrian_holdout_idx[ds] = np.load(cal_dir / 'X_mondrian_holdout_indices.npy')
    for model_name in MODELS_PER_DATASET:
        strict_holdout_probs[(ds, model_name)] = np.load(cal_dir / f'{model_name}_holdout_proba_strict.npy')
        strict_test_probs[(ds, model_name)] = np.load(cal_dir / f'{model_name}_test_proba_strict.npy')

# Load Phase B threshold-calibration sub-slice indices (we'll reuse 08a's split)
sub_split = {}
for ds in DATASETS:
    cal_dir = Path(REPO) / 'calibrators' / ds
    tcal_path = cal_dir / 'X_threshold_calibration_indices.npy'
    teval_path = cal_dir / 'X_threshold_evaluation_indices.npy'
    if tcal_path.exists() and teval_path.exists():
        sub_split[ds] = {
            'tcal_idx_within_holdout': np.load(tcal_path),
            'teval_idx_within_holdout': np.load(teval_path),
        }
    else:
        # Generate fresh if 08a didn't run
        holdout_idx = mondrian_holdout_idx[ds]
        y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
        y_holdout = y_calib_full[holdout_idx]
        n = len(y_holdout)
        rng = np.random.RandomState(SEED + 1)
        class_counts = pd.Series(y_holdout).value_counts().to_dict()
        tcal_mask = np.zeros(n, dtype=bool)
        for c, cnt in class_counts.items():
            idx_c = np.where(y_holdout == c)[0]
            rng.shuffle(idx_c)
            n_tcal = int(round(len(idx_c) * 0.5)) if cnt >= 10 else len(idx_c)
            tcal_mask[idx_c[:n_tcal]] = True
        sub_split[ds] = {
            'tcal_idx_within_holdout': np.where(tcal_mask)[0],
            'teval_idx_within_holdout': np.where(~tcal_mask)[0],
        }
        np.save(tcal_path, sub_split[ds]['tcal_idx_within_holdout'])
        np.save(teval_path, sub_split[ds]['teval_idx_within_holdout'])

    print(f'{ds}: tcal={len(sub_split[ds]["tcal_idx_within_holdout"])}, '
          f'teval={len(sub_split[ds]["teval_idx_within_holdout"])}')

nsl_kdd_v2: tcal=2518, teval=2519
unsw_nb15_v2: tcal=2708, teval=2707
cic_ids2017_v2: tcal=4000, teval=4000


In [5]:
# Load c2 stability (unchanged)
df_stab = pd.read_csv(Path(REPO) / 'results' / 'tables' / 'stability_v2_per_sample_jaccard.csv')
worst = df_stab.groupby(['dataset', 'model', 'sample_position'])['jaccard_top10'].min().reset_index()
worst.rename(columns={'jaccard_top10': 'worst_jaccard'}, inplace=True)
c2_lookup = {}
for (ds, m), g in worst.groupby(['dataset', 'model']):
    c2_lookup[(ds, m)] = g.sort_values('sample_position')['worst_jaccard'].values.astype(np.float32)
print(f'c2 lookup for {len(c2_lookup)} cells')

c2 lookup for 18 cells


In [6]:
# Build per-cell features dataset: 90 rows max
# Each row: (dataset, model, predicted_class) cell with:
#   - mondrian_threshold (from Phase A, fitted on full Mondrian holdout)
#   - cliff_fraction (computed on tcal partition only)
#   - n_calib (Mondrian's n_per_predicted_class)
#   - per_cell_pearson_on_tcal (per-sample SCTS-correctness Pearson within this cell, computed on tcal samples)
#   - n_samples_in_tcal (how many tcal samples fall in this cell)

print('Building per-cell features dataset...')

cell_rows = []
for ds in DATASETS:
    holdout_idx = mondrian_holdout_idx[ds]
    tcal_idx_within = sub_split[ds]['tcal_idx_within_holdout']
    y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
    y_holdout = y_calib_full[holdout_idx]
    y_tcal = y_holdout[tcal_idx_within]

    for model_name in MODELS_PER_DATASET:
        p_holdout = strict_holdout_probs[(ds, model_name)]
        p_tcal = p_holdout[tcal_idx_within]
        y_pred_tcal = p_tcal.argmax(axis=1)

        # Mondrian thresholds on full Mondrian holdout (Phase A protocol)
        mthresh, fallback, n_per = mondrian_conformal_thresholds(
            p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN
        )

        # Per-cell SCTS computation on tcal partition for Pearson
        # Compute c1, c2, c3 on tcal samples (we need c2 lookup mapped to tcal positions — c2 is sample-level)
        # c2 lookup is indexed by canonical 1000 position, not Mondrian holdout. We can't easily get c2 on tcal samples.
        # SOLUTION: compute Pearson between p_max (a c1 proxy) and correctness on tcal cells.
        # This is a reasonable proxy for SCTS-correctness Pearson since c1 typically dominates.

        c1_tcal = p_tcal[np.arange(len(y_tcal)), y_pred_tcal].astype(np.float32)
        correct_tcal = (y_pred_tcal == y_tcal).astype(int)

        # Cliff fractions on tcal partition (per predicted class)
        scores_tcal = 1.0 - p_tcal[np.arange(len(y_tcal)), y_tcal]
        for pc in range(5):
            mask_pc = y_pred_tcal == pc
            n_in_tcal = int(mask_pc.sum())
            cliff = float((scores_tcal[mask_pc] >= CLIFF_THRESH).mean()) if n_in_tcal > 0 else float('nan')

            # Per-cell Pearson on tcal (using c1 proxy for SCTS)
            if n_in_tcal >= MIN_SAMPLES_FOR_PEARSON:
                c1_cell = c1_tcal[mask_pc]
                correct_cell = correct_tcal[mask_pc]
                if c1_cell.std() > 1e-9 and correct_cell.std() > 1e-9:
                    pearson = float(np.corrcoef(c1_cell, correct_cell)[0, 1])
                    if np.isnan(pearson):
                        pearson = None
                else:
                    pearson = None
            else:
                pearson = None

            cell_rows.append({
                'dataset': ds,
                'model': model_name,
                'predicted_class_idx': pc,
                'predicted_class': CLASS_NAMES_5[pc],
                'mondrian_threshold': float(mthresh[pc]),
                'is_fallback': pc in fallback,
                'cliff_fraction': cliff,
                'n_calib_full_holdout': int(n_per[pc]),
                'n_samples_in_tcal': n_in_tcal,
                'pearson_on_tcal': pearson,
            })

df_cells = pd.DataFrame(cell_rows)
print(f'\nTotal cells: {len(df_cells)} (expect 90)')
print(f'Cells with valid Pearson on tcal: {df_cells["pearson_on_tcal"].notna().sum()}')
print(f'  Per-dataset breakdown:')
print(df_cells.groupby('dataset')['pearson_on_tcal'].apply(lambda x: f'{x.notna().sum()}/{len(x)}').to_dict())
print(f'\nMean per-cell Pearson on tcal (overall): {df_cells["pearson_on_tcal"].mean():+.3f}')
print(f'\nPer-cell Pearson on tcal — distribution:')
print(df_cells['pearson_on_tcal'].describe().round(3))

Building per-cell features dataset...

Total cells: 90 (expect 90)
Cells with valid Pearson on tcal: 50
  Per-dataset breakdown:
{'cic_ids2017_v2': '16/30', 'nsl_kdd_v2': '10/30', 'unsw_nb15_v2': '24/30'}

Mean per-cell Pearson on tcal (overall): +0.478

Per-cell Pearson on tcal — distribution:
count    50.000
mean      0.478
std       0.225
min       0.007
25%       0.355
50%       0.417
75%       0.612
max       1.000
Name: pearson_on_tcal, dtype: float64


In [7]:
# Grid search: maximize GREEN-RED Pearson gap subject to constraints
def apply_flag(df, t_green_lo, cliff_green_hi, n_green_lo):
    """Apply flag rules to per-cell df and return flag array."""
    s1 = np.where(
        (df['mondrian_threshold'] >= T_RED_HI) | (df['mondrian_threshold'] < T_RED_LO),
        'red',
        np.where(
            (df['mondrian_threshold'] >= T_GREEN_HI_FIXED) | (df['mondrian_threshold'] <= t_green_lo),
            'amber', 'green'
        )
    )
    cf = df['cliff_fraction'].fillna(1.0).values
    s2 = np.where(cf >= CLIFF_RED_LO, 'red',
                  np.where(cf >= cliff_green_hi, 'amber', 'green'))
    nc = df['n_calib_full_holdout'].values
    s3 = np.where(nc < N_RED_HI, 'red',
                  np.where(nc < n_green_lo, 'amber', 'green'))
    flags = np.where((s1 == 'red') | (s2 == 'red') | (s3 == 'red'), 'red',
                     np.where((s1 == 'amber') | (s2 == 'amber') | (s3 == 'amber'), 'amber', 'green'))
    return flags

def score_combo(df, t_green_lo, cliff_green_hi, n_green_lo):
    flags = apply_flag(df, t_green_lo, cliff_green_hi, n_green_lo)
    valid_mask = df['pearson_on_tcal'].notna().values

    green_mask = (flags == 'green') & valid_mask
    red_mask = (flags == 'red') & valid_mask

    n_green = int(green_mask.sum())
    n_red = int(red_mask.sum())
    n_amber = int(((flags == 'amber') & valid_mask).sum())
    n_valid = int(valid_mask.sum())

    if n_green == 0 or n_red == 0:
        gap = float('nan')
        green_pearson = float('nan')
        red_pearson = float('nan')
    else:
        green_pearson = float(df.loc[green_mask, 'pearson_on_tcal'].mean())
        red_pearson = float(df.loc[red_mask, 'pearson_on_tcal'].mean())
        gap = green_pearson - red_pearson

    # Constraints — all flags (not just valid-pearson ones) for cell-count constraints
    n_green_all = int((flags == 'green').sum())
    n_red_all = int((flags == 'red').sum())
    satisfies_constraints = (
        n_green_all >= MIN_GREEN_CELLS and
        n_red_all >= MIN_RED_CELLS and
        n_valid >= MIN_VALID_PEARSON_CELLS
    )

    return {
        'green_pearson': green_pearson,
        'red_pearson': red_pearson,
        'gap': gap,
        'n_green': n_green_all,
        'n_amber': n_amber,
        'n_red': n_red_all,
        'n_valid_pearson_cells': n_valid,
        'satisfies_constraints': satisfies_constraints,
    }

print('Running grid search with Pearson-gap objective...')
grid_records = []
for t_green_lo, cliff_green_hi, n_green_lo in product(GRID_T_GREEN_LO, GRID_CLIFF_GREEN_HI, GRID_N_GREEN_LO):
    res = score_combo(df_cells, t_green_lo, cliff_green_hi, n_green_lo)
    res.update({
        'T_GREEN_LO': t_green_lo,
        'CLIFF_GREEN_HI': cliff_green_hi,
        'N_GREEN_LO': n_green_lo,
    })
    grid_records.append(res)

df_grid = pd.DataFrame(grid_records)
out_dir = Path(REPO) / 'results' / 'tables'
df_grid.to_csv(out_dir / 'phase_b_v2_grid_search.csv', index=False)
print(f'Grid search complete: {len(df_grid)} combos')

# Among combos satisfying constraints, pick max gap
valid_combos = df_grid[df_grid['satisfies_constraints']].copy()
print(f'\nCombos satisfying all constraints: {len(valid_combos)} / 150')

if len(valid_combos) > 0:
    best_idx = valid_combos['gap'].idxmax()
    best = valid_combos.loc[best_idx]
    print(f'\n*** BEST COMBO (constrained) ***')
    print(f'  T_GREEN_LO = {best["T_GREEN_LO"]}')
    print(f'  CLIFF_GREEN_HI = {best["CLIFF_GREEN_HI"]}')
    print(f'  N_GREEN_LO = {best["N_GREEN_LO"]}')
    print(f'  Pearson gap = {best["gap"]:+.4f}')
    print(f'  GREEN Pearson = {best["green_pearson"]:+.4f} (n={best["n_green"]} cells)')
    print(f'  RED Pearson = {best["red_pearson"]:+.4f} (n={best["n_red"]} cells)')

    print(f'\nTop 5 combos by gap (constrained):')
    top5 = valid_combos.nlargest(5, 'gap')
    print(top5[['T_GREEN_LO', 'CLIFF_GREEN_HI', 'N_GREEN_LO', 'gap',
                'green_pearson', 'red_pearson', 'n_green', 'n_amber', 'n_red']].to_string(index=False))

    # Compare to v2 default thresholds
    v2_res = score_combo(df_cells, 0.05, 0.05, 100)
    print(f'\nFor comparison, v2 defaults (T_GREEN_LO=0.05, CLIFF_GREEN_HI=0.05, N_GREEN_LO=100):')
    print(f'  Gap = {v2_res["gap"]:+.4f}, GREEN={v2_res["n_green"]}, AMBER={v2_res["n_amber"]}, RED={v2_res["n_red"]}')
    print(f'  GREEN Pearson = {v2_res["green_pearson"]:+.4f}, RED Pearson = {v2_res["red_pearson"]:+.4f}')

    selected_combo = best
else:
    print('\n*** NO COMBO SATISFIES CONSTRAINTS ***')
    print('Fallback: report unconstrained optimum')
    has_valid_gap = df_grid['gap'].notna()
    if has_valid_gap.any():
        best_idx = df_grid.loc[has_valid_gap, 'gap'].idxmax()
        best = df_grid.iloc[best_idx]
        print(f'Unconstrained best: T_GREEN_LO={best["T_GREEN_LO"]}, CLIFF_GREEN_HI={best["CLIFF_GREEN_HI"]}, '
              f'N_GREEN_LO={best["N_GREEN_LO"]}, gap={best["gap"]:+.4f}')
        selected_combo = best
    else:
        selected_combo = None
        print('No valid combo at all. Phase B v2 is a negative result.')

Running grid search with Pearson-gap objective...
Grid search complete: 150 combos

Combos satisfying all constraints: 0 / 150

*** NO COMBO SATISFIES CONSTRAINTS ***
Fallback: report unconstrained optimum
Unconstrained best: T_GREEN_LO=0.2, CLIFF_GREEN_HI=0.05, N_GREEN_LO=30, gap=-0.1760


In [8]:
# Decision point: did we get a defensible selection?
if selected_combo is not None and selected_combo.get('satisfies_constraints', False):
    PHASE_B_V2_THRESHOLDS = {
        'T_GREEN_LO': float(selected_combo['T_GREEN_LO']),
        'T_GREEN_HI': T_GREEN_HI_FIXED,
        'T_RED_LO': T_RED_LO,
        'T_RED_HI': T_RED_HI,
        'CLIFF_GREEN_HI': float(selected_combo['CLIFF_GREEN_HI']),
        'CLIFF_RED_LO': CLIFF_RED_LO,
        'CLIFF_THRESH': CLIFF_THRESH,
        'N_GREEN_LO': int(selected_combo['N_GREEN_LO']),
        'N_RED_HI': N_RED_HI,
        'objective': 'GREEN-vs-RED gap in mean per-cell SCTS-correctness Pearson on tcal partition',
        'constraints': {
            'min_green_cells': MIN_GREEN_CELLS,
            'min_red_cells': MIN_RED_CELLS,
            'min_valid_pearson_cells': MIN_VALID_PEARSON_CELLS,
            'min_samples_for_pearson_per_cell': MIN_SAMPLES_FOR_PEARSON,
        },
        'selected_gap_on_tcal': float(selected_combo['gap']),
        'green_pearson_on_tcal': float(selected_combo['green_pearson']),
        'red_pearson_on_tcal': float(selected_combo['red_pearson']),
        'cells_distribution': {
            'green': int(selected_combo['n_green']),
            'amber': int(selected_combo['n_amber']),
            'red': int(selected_combo['n_red']),
        },
        'status': 'selected_constrained',
        'selected_at': datetime.now().isoformat(),
    }
    print('Phase B v2 thresholds locked.')
elif selected_combo is not None:
    PHASE_B_V2_THRESHOLDS = {
        'status': 'fallback_unconstrained',
        'note': 'No combo satisfied constraints. Reporting unconstrained optimum for transparency.',
        'T_GREEN_LO': float(selected_combo['T_GREEN_LO']),
        'CLIFF_GREEN_HI': float(selected_combo['CLIFF_GREEN_HI']),
        'N_GREEN_LO': int(selected_combo['N_GREEN_LO']),
        'selected_gap_on_tcal': float(selected_combo['gap']) if not np.isnan(selected_combo['gap']) else None,
    }
    print('Phase B v2: fallback to unconstrained optimum (constraints not satisfied).')
else:
    PHASE_B_V2_THRESHOLDS = {
        'status': 'negative_result',
        'note': 'Grid search produced no valid gap. v2 defaults retained.',
    }
    print('Phase B v2: NEGATIVE RESULT — no combo produced a valid Pearson gap.')

with open(out_dir / 'phase_b_v2_selected_thresholds.json', 'w') as f:
    json.dump(PHASE_B_V2_THRESHOLDS, f, indent=2)
print(json.dumps(PHASE_B_V2_THRESHOLDS, indent=2))

Phase B v2: fallback to unconstrained optimum (constraints not satisfied).
{
  "status": "fallback_unconstrained",
  "note": "No combo satisfied constraints. Reporting unconstrained optimum for transparency.",
  "T_GREEN_LO": 0.2,
  "CLIFF_GREEN_HI": 0.05,
  "N_GREEN_LO": 30,
  "selected_gap_on_tcal": -0.1760452584488117
}


In [9]:
# Sanity check: if we selected thresholds, evaluate gap on teval partition too
if PHASE_B_V2_THRESHOLDS.get('status') == 'selected_constrained':
    # Recompute per-cell Pearson on teval partition
    teval_cell_rows = []
    for ds in DATASETS:
        holdout_idx = mondrian_holdout_idx[ds]
        teval_idx_within = sub_split[ds]['teval_idx_within_holdout']
        y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
        y_holdout = y_calib_full[holdout_idx]
        y_teval = y_holdout[teval_idx_within]

        for model_name in MODELS_PER_DATASET:
            p_holdout = strict_holdout_probs[(ds, model_name)]
            p_teval = p_holdout[teval_idx_within]
            y_pred_teval = p_teval.argmax(axis=1)
            c1_teval = p_teval[np.arange(len(y_teval)), y_pred_teval].astype(np.float32)
            correct_teval = (y_pred_teval == y_teval).astype(int)

            mthresh, fallback, n_per = mondrian_conformal_thresholds(
                p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN
            )
            scores_teval = 1.0 - p_teval[np.arange(len(y_teval)), y_teval]

            for pc in range(5):
                mask_pc = y_pred_teval == pc
                n_in_teval = int(mask_pc.sum())
                cliff = float((scores_teval[mask_pc] >= CLIFF_THRESH).mean()) if n_in_teval > 0 else float('nan')
                if n_in_teval >= MIN_SAMPLES_FOR_PEARSON:
                    c1_cell = c1_teval[mask_pc]
                    correct_cell = correct_teval[mask_pc]
                    if c1_cell.std() > 1e-9 and correct_cell.std() > 1e-9:
                        pearson = float(np.corrcoef(c1_cell, correct_cell)[0, 1])
                        if np.isnan(pearson):
                            pearson = None
                    else:
                        pearson = None
                else:
                    pearson = None

                teval_cell_rows.append({
                    'dataset': ds, 'model': model_name,
                    'predicted_class_idx': pc,
                    'mondrian_threshold': float(mthresh[pc]),
                    'cliff_fraction': cliff,
                    'n_calib_full_holdout': int(n_per[pc]),
                    'pearson_on_teval': pearson,
                })

    df_teval_cells = pd.DataFrame(teval_cell_rows)

    # Apply locked thresholds and measure gap on teval
    teval_flags = apply_flag(df_teval_cells.rename(columns={'pearson_on_teval': 'pearson_on_tcal'}),  # reuse function
                              PHASE_B_V2_THRESHOLDS['T_GREEN_LO'],
                              PHASE_B_V2_THRESHOLDS['CLIFF_GREEN_HI'],
                              PHASE_B_V2_THRESHOLDS['N_GREEN_LO'])
    teval_valid = df_teval_cells['pearson_on_teval'].notna()
    teval_green = (teval_flags == 'green') & teval_valid.values
    teval_red = (teval_flags == 'red') & teval_valid.values

    if teval_green.sum() > 0 and teval_red.sum() > 0:
        gap_teval = (df_teval_cells.loc[teval_green, 'pearson_on_teval'].mean() -
                     df_teval_cells.loc[teval_red, 'pearson_on_teval'].mean())
    else:
        gap_teval = float('nan')

    print(f'Sanity check on threshold-evaluation partition:')
    print(f'  Gap on tcal:  {PHASE_B_V2_THRESHOLDS["selected_gap_on_tcal"]:+.4f}')
    print(f'  Gap on teval: {gap_teval:+.4f}')
    print(f'  Delta: {gap_teval - PHASE_B_V2_THRESHOLDS["selected_gap_on_tcal"]:+.4f}')
    print(f'  (Small delta = robust selection; large delta = overfit)')
else:
    print('Skipping sanity check (no valid selection to verify)')

Skipping sanity check (no valid selection to verify)


In [10]:
# Apply Phase B v2 thresholds to canonical 1000 — only if we have a valid selection
if PHASE_B_V2_THRESHOLDS.get('status') == 'selected_constrained':
    # Cliff data computed on full Mondrian holdout (Phase A protocol consistency)
    cliff_data = {}
    for ds in DATASETS:
        holdout_idx = mondrian_holdout_idx[ds]
        y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
        y_holdout = y_calib_full[holdout_idx]
        for model_name in MODELS_PER_DATASET:
            p_holdout = strict_holdout_probs[(ds, model_name)]
            scores = 1.0 - p_holdout[np.arange(len(y_holdout)), y_holdout]
            y_pred_h = p_holdout.argmax(axis=1)
            for pred_cls in range(5):
                mask = y_pred_h == pred_cls
                n_in = int(mask.sum())
                cf = float('nan') if n_in == 0 else float((scores[mask] >= CLIFF_THRESH).mean())
                cliff_data[(ds, model_name, pred_cls)] = {'cliff_fraction': cf, 'n_pred_class_in_holdout': n_in}

    def flag_threshold_pb2(t):
        if t >= PHASE_B_V2_THRESHOLDS['T_RED_HI'] or t < PHASE_B_V2_THRESHOLDS['T_RED_LO']: return 'red'
        if t >= PHASE_B_V2_THRESHOLDS['T_GREEN_HI'] or t <= PHASE_B_V2_THRESHOLDS['T_GREEN_LO']: return 'amber'
        return 'green'

    def flag_cliff_pb2(f):
        if np.isnan(f): return 'red'
        if f >= PHASE_B_V2_THRESHOLDS['CLIFF_RED_LO']: return 'red'
        if f >= PHASE_B_V2_THRESHOLDS['CLIFF_GREEN_HI']: return 'amber'
        return 'green'

    def flag_support_pb2(n):
        if n < PHASE_B_V2_THRESHOLDS['N_RED_HI']: return 'red'
        if n < PHASE_B_V2_THRESHOLDS['N_GREEN_LO']: return 'amber'
        return 'green'

    def combine_flags(*flags):
        if 'red' in flags: return 'red'
        if 'amber' in flags: return 'amber'
        return 'green'

    health_records_pb2 = []
    for ds in DATASETS:
        holdout_idx = mondrian_holdout_idx[ds]
        y_calib_full = np.load(f'{REPO}/data/processed/{ds}/y_calib_5class.npy')
        y_holdout = y_calib_full[holdout_idx]
        for model_name in MODELS_PER_DATASET:
            p_holdout = strict_holdout_probs[(ds, model_name)]
            mthresh, fb, n_per = mondrian_conformal_thresholds(
                p_holdout, y_holdout, ALPHA_PRIMARY, 5, MIN_CALIB_MONDRIAN
            )
            for pc in range(5):
                t = mthresh[pc]
                cf = cliff_data[(ds, model_name, pc)]['cliff_fraction']
                n_calib_pb = n_per[pc]
                s_t = flag_threshold_pb2(t)
                s_c = flag_cliff_pb2(cf)
                s_s = flag_support_pb2(n_calib_pb)
                overall = combine_flags(s_t, s_c, s_s)
                health_records_pb2.append({
                    'dataset': ds, 'model': model_name,
                    'predicted_class_idx': pc, 'predicted_class': CLASS_NAMES_5[pc],
                    'mondrian_threshold': t, 'is_fallback': pc in fb,
                    'n_calib': n_calib_pb, 'cliff_fraction': cf,
                    'signal_threshold': s_t, 'signal_cliff': s_c, 'signal_support': s_s,
                    'calib_health': overall,
                })

    df_health_pb2 = pd.DataFrame(health_records_pb2)
    df_health_pb2.to_csv(out_dir / 'scts_v2_calib_health_phaseb_v2.csv', index=False)

    print(f'Phase B v2 health table: {len(df_health_pb2)} rows')
    print(f'Class-level: {dict(df_health_pb2["calib_health"].value_counts())}')
    print(f'\nPer-dataset:')
    print(df_health_pb2.groupby(['dataset', 'calib_health']).size().unstack(fill_value=0))
else:
    df_health_pb2 = None
    print('Phase B v2 selection failed. Skipping canonical 1000 evaluation.')

Phase B v2 selection failed. Skipping canonical 1000 evaluation.


In [11]:
if df_health_pb2 is not None:
    df_scts_safe = pd.read_csv(out_dir / 'scts_v2_canonical_strict_safe.csv')
    flag_lookup_pb2 = df_health_pb2.set_index(['dataset', 'model', 'predicted_class_idx'])['calib_health'].to_dict()
    df_scts_safe['calib_health_phaseb_v2'] = df_scts_safe.apply(
        lambda row: flag_lookup_pb2.get((row['dataset'], row['model'], row['pred_class']), 'unknown'),
        axis=1,
    )
    df_scts_safe.to_csv(out_dir / 'scts_v2_canonical_with_health_phaseb_v2.csv', index=False)

    # Per-flag Pearson
    per_flag_pearson_pb2 = {}
    for flag in ['green', 'amber', 'red']:
        sub = df_scts_safe[df_scts_safe['calib_health_phaseb_v2'] == flag]
        per_m = []
        for (ds, m), g in sub.groupby(['dataset', 'model']):
            if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
                p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
                if not np.isnan(p): per_m.append(p)
        per_flag_pearson_pb2[flag] = float(np.mean(per_m)) if per_m else None

    print(f'Sample-level (PhaseB v2): {dict(df_scts_safe["calib_health_phaseb_v2"].value_counts())}')
    print(f'\nPer-flag mean Pearson (PhaseB v2):')
    for f in ['green', 'amber', 'red']:
        p = per_flag_pearson_pb2[f]
        if p is not None:
            print(f'  {f.upper():>5}: {p:+.3f}')

    nsl_r2l = df_scts_safe[(df_scts_safe['dataset'] == 'nsl_kdd_v2') & (df_scts_safe['true_class'] == 3)]
    red_pb2 = int((nsl_r2l['calib_health_phaseb_v2'] == 'red').sum())
    nongreen_pb2 = int((nsl_r2l['calib_health_phaseb_v2'] != 'green').sum())
    n_pb2 = len(nsl_r2l)
    print(f'\n*** NSL R2L RED catch rate (PhaseB v2): {red_pb2}/{n_pb2} = {100*red_pb2/n_pb2:.1f}% ***')
    print(f'*** NSL R2L non-GREEN catch rate (PhaseB v2): {nongreen_pb2}/{n_pb2} = {100*nongreen_pb2/n_pb2:.1f}% ***')
else:
    print('No Phase B v2 selection — headline not computed.')

No Phase B v2 selection — headline not computed.


In [12]:
if df_health_pb2 is not None:
    # Four-way: v2 vs strict-safe vs PhaseB-08a vs PhaseB-v2-08b
    df_v2_scts = pd.read_csv(out_dir / 'scts_v2_canonical.csv')
    df_v2_health = pd.read_csv(out_dir / 'scts_v2_calib_health.csv')
    df_strict_safe_health = pd.read_csv(out_dir / 'scts_v2_calib_health_strict_safe.csv')

    # Load 08a results if they exist
    pb1_path = out_dir / 'scts_v2_calib_health_phaseb.csv'
    if pb1_path.exists():
        df_pb1_health = pd.read_csv(pb1_path)
        has_pb1 = True
    else:
        has_pb1 = False

    print('=' * 100)
    print(f'COMPARISON: v2 vs STRICT-SAFE{" vs PhaseB(08a)" if has_pb1 else ""} vs PhaseB-v2(08b)')
    print('=' * 100)

    v2_cl = dict(df_v2_health['calib_health'].value_counts())
    ss_cl = dict(df_strict_safe_health['calib_health'].value_counts())
    pb2_cl = dict(df_health_pb2['calib_health'].value_counts())
    pb1_cl = dict(df_pb1_health['calib_health'].value_counts()) if has_pb1 else {}

    print(f'\nClass-level flag distribution:')
    for f in ['green', 'amber', 'red']:
        line = f'  {f:>6}: v2={v2_cl.get(f, 0):>3}  strict-safe={ss_cl.get(f, 0):>3}'
        if has_pb1: line += f'  PB-08a={pb1_cl.get(f, 0):>3}'
        line += f'  PB-08b={pb2_cl.get(f, 0):>3}'
        print(line)

    # NSL R2L catch rates
    df_v2_merged = df_v2_scts[(df_v2_scts['dataset'] == 'nsl_kdd_v2') & (df_v2_scts['true_class'] == 3)].merge(
        df_v2_health[['dataset', 'model', 'predicted_class_idx', 'calib_health']],
        left_on=['dataset', 'model', 'pred_class'],
        right_on=['dataset', 'model', 'predicted_class_idx'],
    )
    n_v2 = len(df_v2_merged)
    red_v2 = int((df_v2_merged['calib_health'] == 'red').sum())
    nongreen_v2 = int((df_v2_merged['calib_health'] != 'green').sum())

    df_strict_safe_aug = pd.read_csv(out_dir / 'scts_v2_canonical_with_health_strict_safe.csv')
    nsl_r2l_ss = df_strict_safe_aug[(df_strict_safe_aug['dataset'] == 'nsl_kdd_v2') & (df_strict_safe_aug['true_class'] == 3)]
    red_ss = int((nsl_r2l_ss['calib_health'] == 'red').sum())
    nongreen_ss = int((nsl_r2l_ss['calib_health'] != 'green').sum())
    n_ss = len(nsl_r2l_ss)

    print(f'\n*** NSL R2L RED-flag catch rate ***')
    print(f'  v2:          {red_v2}/{n_v2} = {100*red_v2/n_v2:.1f}%')
    print(f'  strict-safe: {red_ss}/{n_ss} = {100*red_ss/n_ss:.1f}%')
    print(f'  PhaseB v2:   {red_pb2}/{n_pb2} = {100*red_pb2/n_pb2:.1f}%')
    print(f'\n*** NSL R2L non-GREEN catch rate ***')
    print(f'  v2:          {nongreen_v2}/{n_v2} = {100*nongreen_v2/n_v2:.1f}%')
    print(f'  strict-safe: {nongreen_ss}/{n_ss} = {100*nongreen_ss/n_ss:.1f}%')
    print(f'  PhaseB v2:   {nongreen_pb2}/{n_pb2} = {100*nongreen_pb2/n_pb2:.1f}%')

    # Per-flag mean Pearson (strict-safe and v2 from JSONs)
    with open(out_dir / 'scts_v2_health_summary.json') as f:
        v2_hsum = json.load(f)
    with open(out_dir / 'scts_v2_health_summary_strict.json') as f:
        ss_hsum = json.load(f)

    print(f'\nPer-flag mean Pearson:')
    for f in ['green', 'amber', 'red']:
        v_p = v2_hsum['summary_by_flag'][f]['mean_pearson_per_model']
        s_p = ss_hsum['summary_by_flag'][f]['mean_pearson_per_model']
        pb2_p = per_flag_pearson_pb2[f]
        print(f'  {f.upper():>5}: v2={v_p:+.3f}, strict-safe={s_p:+.3f}, PhaseB-v2={pb2_p:+.3f}')

    # Save the comparison
    comparison_rows = []
    for ds in DATASETS:
        for model_name in MODELS_PER_DATASET:
            for pc in range(5):
                v2_m = df_v2_health[(df_v2_health['dataset'] == ds) & (df_v2_health['model'] == model_name) & (df_v2_health['predicted_class_idx'] == pc)]
                ss_m = df_strict_safe_health[(df_strict_safe_health['dataset'] == ds) & (df_strict_safe_health['model'] == model_name) & (df_strict_safe_health['predicted_class_idx'] == pc)]
                pb2_m = df_health_pb2[(df_health_pb2['dataset'] == ds) & (df_health_pb2['model'] == model_name) & (df_health_pb2['predicted_class_idx'] == pc)]
                comparison_rows.append({
                    'dataset': ds, 'model': model_name, 'predicted_class': CLASS_NAMES_5[pc],
                    'flag_v2': v2_m['calib_health'].iloc[0] if len(v2_m) else 'na',
                    'flag_strict_safe': ss_m['calib_health'].iloc[0] if len(ss_m) else 'na',
                    'flag_phaseb_v2': pb2_m['calib_health'].iloc[0] if len(pb2_m) else 'na',
                })
    df_4way = pd.DataFrame(comparison_rows)
    df_4way.to_csv(out_dir / 'phase_b_v2_three_way_comparison.csv', index=False)
    print(f'\nSaved phase_b_v2_three_way_comparison.csv ({len(df_4way)} rows)')
else:
    print('Phase B v2 did not produce a valid selection. Recommending fallback to v2 defaults or Option A (commit Phase B as negative result).')

Phase B v2 did not produce a valid selection. Recommending fallback to v2 defaults or Option A (commit Phase B as negative result).


In [13]:
if df_health_pb2 is not None:
    print('=' * 70)
    print('BOOTSTRAP CIs ON PHASE B v2 (B=1000)')
    print('=' * 70)

    rng_boot = np.random.RandomState(BOOTSTRAP_SEED)
    boot_records = []

    # Per-flag mean Pearson
    for flag in ['green', 'amber', 'red']:
        sub = df_scts_safe[df_scts_safe['calib_health_phaseb_v2'] == flag]
        per_m = []
        for (ds, m), g in sub.groupby(['dataset', 'model']):
            if g['scts'].std() > 1e-9 and g['correct'].std() > 1e-9 and len(g) > 5:
                p = float(np.corrcoef(g['scts'], g['correct'])[0, 1])
                if not np.isnan(p): per_m.append(p)
        if len(per_m) < 2: continue
        arr = np.array(per_m)
        n = len(arr)
        boots = np.array([arr[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
        boot_records.append({
            'protocol': 'phaseb_v2', 'metric': 'per_flag_mean_pearson', 'flag': flag,
            'point': float(arr.mean()),
            'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)),
        })

    # NSL R2L catch rates
    is_red = (nsl_r2l['calib_health_phaseb_v2'] == 'red').astype(int).values
    is_nongreen = (nsl_r2l['calib_health_phaseb_v2'] != 'green').astype(int).values

    for name, arr in [('r2l_red_catch_rate', is_red), ('r2l_nongreen_catch_rate', is_nongreen)]:
        n = len(arr)
        boots = np.array([arr[rng_boot.randint(0, n, n)].mean() for _ in range(N_BOOTSTRAP)])
        k = int(arr.sum())
        p_hat = k / n
        z = 1.96
        denom = 1 + z**2/n
        centre = (p_hat + z**2/(2*n)) / denom
        halfw = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n*n)) / denom
        boot_records.append({
            'protocol': 'phaseb_v2', 'metric': name, 'flag': 'n/a',
            'point': float(p_hat),
            'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)),
            'wilson_ci_low': float(centre - halfw),
            'wilson_ci_high': float(centre + halfw),
            'n_total': int(n), 'n_red': int(k),
        })

    df_boot_pb2 = pd.DataFrame(boot_records)
    df_boot_pb2.to_csv(out_dir / 'bootstrap_cis_phaseb_v2.csv', index=False)

    print(f'Bootstrap records: {len(df_boot_pb2)}')
    for _, row in df_boot_pb2.iterrows():
        if row['metric'] == 'per_flag_mean_pearson':
            print(f'  {row["flag"].upper()} Pearson: {row["point"]:+.3f} [{row["ci_low"]:+.3f}, {row["ci_high"]:+.3f}]')
        else:
            print(f'  {row["metric"]}: {row["point"]*100:.1f}% bootstrap [{row["ci_low"]*100:.1f}%, {row["ci_high"]*100:.1f}%]')
else:
    print('No selection — skipping bootstrap.')

No selection — skipping bootstrap.


In [14]:
# Write findings doc — different content if selection succeeded vs failed
if df_health_pb2 is not None and PHASE_B_V2_THRESHOLDS.get('status') == 'selected_constrained':
    status_msg = 'POSITIVE: Phase B v2 produced a defensible selection.'
elif PHASE_B_V2_THRESHOLDS.get('status') == 'fallback_unconstrained':
    status_msg = 'PARTIAL: constraints not satisfied; fallback to unconstrained optimum reported.'
else:
    status_msg = 'NEGATIVE: grid search produced no valid Pearson gap. v2 defaults remain best choice.'

findings = f"""# Phase B v2 Findings — Pearson-Gap Objective

**Date**: {datetime.now().strftime('%Y-%m-%d')}
**Status**: {status_msg}

## Background

Phase B (notebook 08a) tried to pre-register health-flag thresholds using F1 against misclassification. It failed because misclassification rates on the Mondrian holdout were too low (0.3% NSL, 1.2% CIC) for F1 to have signal.

Phase B v2 (this notebook, 08b) retries with a different objective: maximize the GREEN-vs-RED gap in per-cell SCTS-correctness Pearson, subject to constraints (≥10 GREEN cells, ≥10 RED cells, ≥60 valid-Pearson cells).

## Result

{status_msg}

"""

if PHASE_B_V2_THRESHOLDS.get('status') == 'selected_constrained':
    findings += f"""### Selected thresholds (locked)

- T_GREEN_LO = {PHASE_B_V2_THRESHOLDS['T_GREEN_LO']}
- CLIFF_GREEN_HI = {PHASE_B_V2_THRESHOLDS['CLIFF_GREEN_HI']}
- N_GREEN_LO = {PHASE_B_V2_THRESHOLDS['N_GREEN_LO']}

GREEN-RED gap on tcal: {PHASE_B_V2_THRESHOLDS['selected_gap_on_tcal']:+.4f}
- GREEN cells ({PHASE_B_V2_THRESHOLDS['cells_distribution']['green']}): mean Pearson = {PHASE_B_V2_THRESHOLDS['green_pearson_on_tcal']:+.4f}
- RED cells ({PHASE_B_V2_THRESHOLDS['cells_distribution']['red']}): mean Pearson = {PHASE_B_V2_THRESHOLDS['red_pearson_on_tcal']:+.4f}

### Compared to v2 defaults

v2 used T_GREEN_LO=0.05, CLIFF_GREEN_HI=0.05, N_GREEN_LO=100. The Phase B v2 selection differs by [TODO: fill in based on actual output].
"""
else:
    findings += f"""### What this means

The flag's threshold space is genuinely flat with respect to the per-cell Pearson gap. v2's defaults (T_GREEN_LO=0.05, CLIFF_GREEN_HI=0.05, N_GREEN_LO=100) are best understood as interpretable boundary defaults from calibration literature, not in-sample tuned hyperparameters.

The NSL R2L catch rate of 78.8% (Phase A strict-safe) remains the headline operational metric.

Phase B (both 08a and 08b) serves as a sensitivity analysis demonstrating that the flag's behavior does not depend sensitively on threshold choices within the explored grid.
"""

findings += f"""

## Files

- `notebooks/08b_phase_b_v2_pearson_gap.ipynb`
- `results/tables/phase_b_v2_grid_search.csv` — full grid with Pearson gaps and constraints
- `results/tables/phase_b_v2_selected_thresholds.json` — selection record
- `results/tables/scts_v2_calib_health_phaseb_v2.csv` (if selection succeeded)
- `results/tables/scts_v2_canonical_with_health_phaseb_v2.csv` (if selection succeeded)
- `results/tables/phase_b_v2_three_way_comparison.csv` (if selection succeeded)
- `results/tables/bootstrap_cis_phaseb_v2.csv` (if selection succeeded)
"""

docs_dir = Path(REPO) / 'docs'
with open(docs_dir / 'phase_b_v2_findings.md', 'w') as f:
    f.write(findings)
print(f'Wrote: docs/phase_b_v2_findings.md')

Wrote: docs/phase_b_v2_findings.md


In [ ]:
# Commit + push
os.chdir(REPO)
!git status --short

print('\n>>> Staging Phase B v2 files...')
!git add notebooks/08b_phase_b_v2_pearson_gap.ipynb
!git add results/tables/phase_b_v2_grid_search.csv
!git add results/tables/phase_b_v2_selected_thresholds.json
!git add results/tables/scts_v2_calib_health_phaseb_v2.csv 2>/dev/null
!git add results/tables/scts_v2_canonical_with_health_phaseb_v2.csv 2>/dev/null
!git add results/tables/phase_b_v2_three_way_comparison.csv 2>/dev/null
!git add results/tables/bootstrap_cis_phaseb_v2.csv 2>/dev/null
!git add docs/phase_b_v2_findings.md

!git status --short
!git commit -m "Phase B v2: Pearson-gap objective for pre-registered flag thresholds (08b retry of 08a's failed F1 objective). Result documented in docs/phase_b_v2_findings.md."
!git push origin main

print('\n>>> Drive saved + git pushed? Confirm before next step.')